# Finnish ASR Baseline: Parakeet TDT 0.6B v3

Team Tokens — COE025 Natural Language Processing, Project 3.

This experiment evaluates `nvidia/parakeet-tdt-0.6b-v3`, a pretrained multilingual Token-and-Duration Transducer (TDT), without additional training by the team. It is a baseline, not a team fine-tuning contribution. The architecture is reported as TDT rather than plain RNN-T.

**Checkpoint revision:** `541d1f99c6b0c3cd0b11a95167540bb8edefd82b`.

The [official model card](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3) lists Finnish among the supported languages. The model identifies the language automatically; no source-language or target-language prompt is supplied.

The configured evaluation uses CUDA, float32 precision, and a batch size of 1. The runtime records the actual GPU and software versions. The pretrained checkpoint's decoding configuration is preserved and exported with the results.


## 1. Dependencies

The environment uses NeMo 3.0.0 and Transformers 4.57.6. The existing PyTorch version is retained through an installation constraint. The setup checks Python, PyTorch, and CUDA availability.


In [1]:
import sys,subprocess
from pathlib import Path
import torch
from packaging.version import Version
assert sys.version_info >= (3,12), 'Python 3.12 veya üzeri Colab çalışma zamanı gerekli.'
assert torch.cuda.is_available(), 'Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç.'
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), 'PyTorch sürümü kontrol edilmeli.'
constraint=Path('/content/ece_nemo_torch_constraint.txt')
constraint.write_text('torch=='+torch.__version__+'\n',encoding='utf8')
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                '-c',str(constraint),'nemo_toolkit[asr]==3.0.0','transformers==4.57.6',
                'jiwer==4.0.0','soundfile==0.13.1','librosa==0.11.0','pyarrow'],check=True)
print('KURULUM TAMAM')


KURULUM TAMAM


## 2. Experiment Files

The embedded archive contains the evaluation scripts, fixed data manifests, and experiment protocol. Its checksum is verified before extraction. Dataset and model revisions are pinned in the protocol.


In [2]:
import base64,hashlib,io,zipfile
PACKAGE=Path('/content/ece_nemo')
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='cff05657770aaf8eae3047d93be28531a65208ea54857ae82f500c9c4183f7ab'
PACKAGE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    assert all((PACKAGE/m.filename).resolve().is_relative_to(PACKAGE.resolve()) for m in archive.infolist())
    archive.extractall(PACKAGE)
print('Dosyalar doğrulandı.')


Dosyalar doğrulandı.


## 3. Environment and Data Validation

Checks cover NeMo imports, available disk space, manifest integrity, the shared evaluator, and reference normalization. Subprocess output is displayed and saved to logs.


In [3]:
import os,datetime,platform,shutil
from collections import deque
MODEL='parakeet'
OUT=Path('/content/Ece_Parakeet_outputs')
OUT.mkdir(parents=True,exist_ok=True)
CACHE='/content/asr_audio'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['WANDB_MODE']='disabled'
os.environ['TOKENIZERS_PARALLELISM']='false'
print('GPU:',torch.cuda.get_device_name(0),'| Python:',platform.python_version())
print('Boş disk:',round(shutil.disk_usage('/content').free/2**30,1),'GB')
assert shutil.disk_usage('/content').free>30*2**30, '30 GB boş disk gerekli; önce dosyaları yedekle.'

def command(args):
    logs=OUT/'logs';logs.mkdir(exist_ok=True)
    log_path=logs/(datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.log')
    tail=deque(maxlen=40)
    with log_path.open('w',encoding='utf8') as log:
        with subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
                              encoding='utf8',errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line)
            status=process.wait()
    if status:
        print('SON HATA SATIRLARI:\n'+''.join(tail))
        raise RuntimeError(f'İşlem durdu. Son hata satırlarını paylaş. Kayıt: {log_path}')

def run(script,*args):command([sys.executable,'-u',PACKAGE/script,*args])
command([sys.executable,'-c','from nemo.collections.asr.models import ASRModel; import nemo; print("NeMo:",nemo.__version__)'])
run('test_protocol.py')
print('HAZIRLIK TAMAM')


GPU: Tesla T4 | Python: 3.13.16
Boş disk: 112.2 GB
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
NeMo: 3.0.0
.....
----------------------------------------------------------------------
Ran 5 tests in 0.338s

OK
HAZIRLIK TAMAM


## 4. Development Smoke Test

The sample consists of the first eight and the eight longest recordings in `dev_cv`, with duplicates removed. It checks the inference pipeline, including longer inputs. These sample scores are not full development-set or test-set results.

Runtime estimates extrapolate the measured inference time and exclude downloads and setup. Empty predictions are retained as recognition errors; an entirely empty development output stops the run.


In [9]:
run('eval_nemo.py','--model',MODEL,'--split','dev_cv','--out',OUT,'--cache',CACHE,'--smoke')


REF: punaisessa mekossa 
HYP: punaisessa mekossa
REF: mutta kun riihessä liikkui 
HYP: mutta kun riihessä liikkui
REF: en nähnyt planeetalla mitään suuttimia joista sade suihkuaisi planeetalle 
HYP: en nähnyt planeetalla mitään suuttimia joista sade suihkuaisi planeetalle
REF: hän tervehti minua iloisesti tavallistaakko hän tokaisi filaitokkiliksi 
HYP: hän tervehti minua iloisesti tavallistaakko hän tokaisi filaitokkiliksi
REF: tänää tehdään nimittäin historiaa 
HYP: tänää tehdään nimittäin historiaa
REF: sen ylitti silta jonka alla oli karppeja 
HYP: sen ylitti silta jonka alla oli karppeja
REF: työvuorossa oleva cassandra avasi oven ja astui sisään tummanruskeat hiukset hulmuten 
HYP: työvuorossa oleva cassandra avasi oven ja astui sisään tummanruskeat hiukset hulmuten
REF: toivottavasti nancy ja raivo ovat turvassa 
HYP: toivottavasti nancy ja raivo ovat turvassa
REF: ne hedelmät kaikki oikeestaa sisäls aika ison annoksen peroksirikkihappoo diego ilmoitti hätääntyneellä äänellä 
HY

## 5. Result Archive

Each archive contains the results available at the time of export, including predictions, scores, progress records, model configurations, logs, and experiment scripts. Audio files and model weights are excluded.


In [5]:
from google.colab import files

def download_results():
    path=Path('/content/Ece_Parakeet_sonuclar_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for source,prefix in [(OUT,''),(PACKAGE,'scripts/ece_nemo')]:
            for p in sorted(source.rglob('*')):
                if p.is_file() and '__pycache__' not in p.parts and p.suffix in {'.py','.json','.jsonl','.csv','.md','.log'}:
                    z.write(p,Path(prefix)/p.relative_to(source))
    print('İndiriliyor:',path)
    files.download(str(path))

download_results()


İndiriliyor: /content/Ece_Parakeet_sonuclar_20261121_124846.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Common Voice Test Set — `test_cv`

The test set contains **1,833 recordings**. Predictions are matched to the fixed manifest by recording ID. Completed predictions can be reused within the same runtime when the recorded environment and protocol match.


In [11]:
assert (OUT/'dev_smoke/scores/parakeet-tdt-0-6b-v3-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_nemo.py','--model',MODEL,'--split','test_cv','--out',OUT,'--cache',CACHE)


Model yükleniyor: nvidia/parakeet-tdt-0.6b-v3
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-11-21 12:55:28 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buf

In [12]:
download_results()


İndiriliyor: /content/Ece_Parakeet_sonuclar_20261121_125854.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Parliament Test Set — `test_parl`

The test set contains **1,913 recordings**. Predictions are matched to the fixed manifest by recording ID. Completed predictions can be reused within the same runtime when the recorded environment and protocol match.


In [13]:
assert (OUT/'dev_smoke/scores/parakeet-tdt-0-6b-v3-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_nemo.py','--model',MODEL,'--split','test_parl','--out',OUT,'--cache',CACHE)


Model yükleniyor: nvidia/parakeet-tdt-0.6b-v3
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-11-21 13:03:09 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buf

In [14]:
download_results()


İndiriliyor: /content/Ece_Parakeet_sonuclar_20261121_130811.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. FLEURS Test Set — `test_fleurs`

The test set contains **916 recordings**. Predictions are matched to the fixed manifest by recording ID. Completed predictions can be reused within the same runtime when the recorded environment and protocol match.


In [15]:
assert (OUT/'dev_smoke/scores/parakeet-tdt-0-6b-v3-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_nemo.py','--model',MODEL,'--split','test_fleurs','--out',OUT,'--cache',CACHE)


Model yükleniyor: nvidia/parakeet-tdt-0.6b-v3
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-11-21 13:09:45 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buf

In [16]:
download_results()


İndiriliyor: /content/Ece_Parakeet_sonuclar_20261121_131235.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Test Completion and Results

All three test sets were completed on a Tesla T4 GPU using float32 precision and a batch size of 1.

| Test set | Recordings | WER (%) | CER (%) |
|---|---:|---:|---:|
| Common Voice (`test_cv`) | 1,833 | 7.17 | 1.37 |
| Parliament (`test_parl`) | 1,913 | 13.28 | 4.09 |
| FLEURS (`test_fleurs`) | 916 | 13.35 | 2.72 |

All 4,662 prediction IDs and references match the fixed manifests. The number of empty predictions is 5; empty predictions are retained and scored rather than excluded. Corpus-level WER and CER were independently recomputed from the exported prediction files and agree with the recorded scores.

The following cell verifies the expected test-set counts and displays the recorded results. The model is a pretrained TDT baseline, not a team fine-tuned model or a plain RNN-T result.


In [17]:
import csv
score_file=OUT/'results/scores/parakeet-tdt-0-6b-v3-baseline.csv'
with score_file.open(encoding='utf8',newline='') as f: rows=list(csv.DictReader(f))
expected={'test_cv':1833,'test_parl':1913,'test_fleurs':916}
assert len(rows)==3 and {r['test_set'] for r in rows}==set(expected), 'Üç test henüz tamamlanmadı.'
for r in rows:
    assert int(r['clips'])==expected[r['test_set']]
    print(r['test_set'],'| kayıt:',r['clips'],'| WER:',r['wer'],'| CER:',r['cer'])
print('ÜÇ TEST TAMAM. Son ZIP ve çalıştırılmış notebook dosyasını sakla.')
download_results()


test_cv | kayıt: 1833 | WER: 7.17 | CER: 1.37
test_parl | kayıt: 1913 | WER: 13.28 | CER: 4.09
test_fleurs | kayıt: 916 | WER: 13.35 | CER: 2.72
ÜÇ TEST TAMAM. Son ZIP ve çalıştırılmış notebook dosyasını sakla.
İndiriliyor: /content/Ece_Parakeet_sonuclar_20261121_131537.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Evaluation Protocol and Limitations

Each test set is reported separately. Scores and per-recording predictions use the team's standard `results/scores` and `results/predictions` formats. The original pretrained baseline is separate from systems fine-tuned by the team.

References use the manifest's existing `text` values. The shared `asr_eval.py` normalizer is applied once to each raw model prediction. Corpus-level WER and CER are computed with the same evaluator used by the other systems. Missing or duplicate recordings prevent a complete test result from being accepted.

Inference timing covers the complete NeMo transcription call on in-memory audio after a development warm-up, with CUDA synchronization. It includes temporary data-loader setup, feature extraction, GPU transfer, forward computation, and decoding. WAV reading, model loading, downloads, scoring, and logging are excluded. FP32/batch-1 measurements are not a controlled speed comparison with runs using different hardware, precision, or batch sizes.

Recording-level and speaker-level overlap between the publisher's training data and the team's test sets has not been verified. Training-test disjointness is therefore not established for this pretrained checkpoint.

Progress records are stored on the Colab runtime's local disk and are lost if that runtime is deleted. Automatic restoration from a downloaded archive is not implemented.
